# Week 6: Inspecting logits

This notebook is runnable without the instructor checkpoint. Configure the runtime separately to inspect real Chinese, Nôm, and detector-box crops.

## From raw scores to the next token

At each decoder step, the model produces one **logit** (raw, unnormalised score) for every token in the vocabulary. For logits $z = (z_1, \ldots, z_V)$, softmax converts each raw score into a probability:

$$\operatorname{softmax}(z)_i = \frac{e^{z_i}}{\sum_{j=1}^{V} e^{z_j}}.$$

The probabilities are non-negative and sum to 1. In code, subtract the row maximum $m = \max_j z_j$ first to avoid very large exponentials. This is numerically stable because $\operatorname{softmax}(z) = \operatorname{softmax}(z - m)$. Subtracting the same value also means the winning token is unchanged: `argmax(logits) == argmax(probabilities)`.

The next cell makes every intermediate value visible: raw logits → maximum-shifted logits → exponentials → normalized softmax probabilities → the argmax token. Read one row at a time; each row represents a different decoder step.

In [1]:
import numpy as np

vocabulary = ['<pad>', '<sos>', '<eos>', '⿰', '木']
logits = np.array([[-2.0, -3.0, -4.0, 3.0, 1.0], [-1.0, -3.0, 4.0, 0.5, 1.0]])
for step, row in enumerate(logits):
    row_max = row.max()
    shifted = row - row_max
    exponentials = np.exp(shifted)
    probability = exponentials / exponentials.sum()
    token_id = int(row.argmax())

    print(f'\nstep={step}')
    print(f'  raw logits:        {row}')
    print(f'  subtract max ({row_max:.1f}): {shifted}')
    print(f'  exp(shifted):      {np.array2string(exponentials, precision=4)}')
    print(f'  softmax:           {np.array2string(probability, precision=4)} (sum={probability.sum():.4f})')
    print(f'  argmax: {token_id}:{vocabulary[token_id]} (probability={probability[token_id]:.4f})')

probabilities = np.array([
    np.exp(row - row.max()) / np.exp(row - row.max()).sum()
    for row in logits
])

assert logits.shape == (2, len(vocabulary))
assert int(logits[0].argmax()) == 3
assert int(logits[1].argmax()) == 2
assert np.allclose(probabilities.sum(axis=1), 1.0)
assert np.array_equal(logits.argmax(axis=1), probabilities.argmax(axis=1))
print('Tiny logits inspection passed.')


step=0
  raw logits:        [-2. -3. -4.  3.  1.]
  subtract max (3.0): [-5. -6. -7.  0. -2.]
  exp(shifted):      [0.0067 0.0025 0.0009 1.     0.1353]
  softmax:           [0.0059 0.0022 0.0008 0.873  0.1182] (sum=1.0000)
  argmax: 3:⿰ (probability=0.8730)

step=1
  raw logits:        [-1.  -3.   4.   0.5  1. ]
  subtract max (4.0): [-5.  -7.   0.  -3.5 -3. ]
  exp(shifted):      [6.7379e-03 9.1188e-04 1.0000e+00 3.0197e-02 4.9787e-02]
  softmax:           [6.1957e-03 8.3851e-04 9.1940e-01 2.7767e-02 4.5798e-02] (sum=1.0000)
  argmax: 2:<eos> (probability=0.9194)
Tiny logits inspection passed.


## Real checkpoint exercise

After receiving the private YAML and compatible checkpoint, load `RecognitionConfig` and `RecognizerRuntime`. Use the crop iterator from your team implementation, then inspect `teacher_forced_logits(crops, prefixes)`. Record the prefix and top-*k* tokens for provided Chinese and Nôm crops and one Week 4 detector page. Do not call `greedy_search` in this notebook.